# BM25 vs Dense retrieval 비교

기본 RAG에서는 보통 Dense Retrieval, 즉 임베딩 벡터 기반 검색만 사용한다. 하지만 Dense Retrieval이 항상 모든 질문에 가장 좋은 것은 아니다. 특정 키워드, 고유명사, 숫자, 버전처럼 정확한 단어가 중요한 질문에서는 BM25 같은 키워드 기반 검색이 더 유리할 수 있다.

In [1]:
%pip install rank_bm25 konlpy langchain langchain_openai langchain_pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 환경설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [2]:
import pandas as pd

document_df = pd.read_csv("data/documents.csv")
queries_df = pd.read_csv("data/queries.csv")
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=2
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=2
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=2


## BM25기반 검색기

BM25는 **문서 집합에서 쿼리와 각 문서의 연관성 점수**를 계산하는 대표적인 키워드 기반 랭킹 알고리즘이다. 주로 정보 검색, 추천 시스템, 검색 엔진 등에서 사용된다.

BM25는 질문과 문서에 같은 단어가 얼마나 중요하게 등장하는지를 기준으로 문서를 찾는다. 따라서 특정 키워드, 고유명사, 버전명, 숫자 등이 중요한 질문에서 강점을 가진다.

반면 사용자가 문서와 다른 표현으로 질문하면 약할 수 있다. 예를 들어 문서에는 “당일치기”라고 되어 있는데 사용자가 “하루 다녀올 만한 곳”이라고 질문하면 키워드가 직접 일치하지 않아 점수가 낮아질 수 있다.

- **토큰화 품질**이 BM25의 성능에 큰 영향을 미친다.
- **문서와 쿼리의 토크나이저**는 동일해야 한다.
- **점수(score)**가 높을수록 쿼리와 문서의 연관성이 높다.

한국어는 공백만으로 단어가 잘 분리되지 않기 때문에, 여기서는 `Okt` 형태소 분석기를 사용해 문서와 질문을 같은 방식으로 토큰화한다.

**주요 메소드**

| 메소드                       | 설명                                         |
|-----------------------------|----------------------------------------------|
| `BM25Okapi(tokenized_corpus)` | 토큰화된 문서 집합으로 BM25 모델 생성        |
| `get_scores(tokenized_query)` | 쿼리에 대해 각 문서의 BM25 점수 반환         |
| `get_top_n(tokenized_query, corpus, n)` | 점수 기준 상위 n개 문서 반환      |

**파라미터 설명**

- **k1**: 단어 빈도(TF)에 대한 가중치 (기본값: 1.5)
- **b**: 문서 길이 보정 파라미터 (기본값: 0.75)
- **epsilon**: 음수 IDF 방지 하한값 (기본값: 0.25)
- 파라미터는 BM25Okapi 생성 시 지정 가능하다.

**BM25 기본 공식**

BM25는 **쿼리 Q와 문서 D의 관련성 점수**를 다음 공식으로 계산한다:

$$
\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

여기서:
- $q_i$: 쿼리의 i번째 단어
- $f(q_i, D)$: 문서 D에서 $q_i$의 등장 횟수 (TF)
- $|D|$: 문서 D의 길이 (단어 수)
- $\text{avgdl}$: 전체 문서 집합의 평균 길이
- $k_1$, $b$: 조정 파라미터 (기본값: $k_1=1.5$, $b=0.75$)

In [3]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt()
tokenized_docs = [okt.morphs(content) for content in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    """
    BM25로 질문과 관련 있는 상위 문서 ID를 반환한다.
    """
    # 질문도 문서와 동일한 형태소 단위로 토큰화 한다.
    query_token = okt.morphs(query)

    # 각 문서에 대한 BM25 점수를 계산
    scores = bm25.get_scores(query_token)

    # 점수가 높은 문서 순서대로 정렬
    sorted_idx = sorted(range(len(scores)), key=lambda i:scores[i], reverse=True)

    # 문서 본문이 아닌 문서 id만 반환
    ranked_docs = [document_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs

bm25_search('제주도 관광 명소')

['D1', 'D12', 'D2', 'D3', 'D4']

In [4]:
document_df[document_df['doc_id'] == 'D12']['content'].values

<ArrowStringArray>
['서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, 양평 두물머리, 용인 에버랜드 등이 있습니다. 기차·버스 노선이 잘 발달되어 있어 대중교통으로 이동이 편리하며, 차가 있다면 경춘고속도로를 이용해 접근성이 좋습니다. 사전 관광 예약 앱(예: 야놀자, 쿠팡트래블)에서도 할인 혜택을 확인할 수 있습니다.']
Length: 1, dtype: str

## 모든 질문에 대해서 BM25 검색 실행

In [5]:
bm25_results = {}

for idx, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']

    bm25_results[qid] = bm25_search(query_text, top_k=5)

# 각 질문에 대해 BM25 상위 5개 문서를 저장
bm25_results

{'Q1': ['D1', 'D2', 'D3', 'D4', 'D5'],
 'Q2': ['D13', 'D2', 'D10', 'D27', 'D9'],
 'Q3': ['D3', 'D1', 'D9', 'D2', 'D19'],
 'Q4': ['D4', 'D30', 'D2', 'D5', 'D9'],
 'Q5': ['D5', 'D8', 'D24', 'D15', 'D20'],
 'Q6': ['D6', 'D14', 'D25', 'D10', 'D3'],
 'Q7': ['D25', 'D30', 'D7', 'D14', 'D5'],
 'Q8': ['D8', 'D29', 'D7', 'D12', 'D1'],
 'Q9': ['D9', 'D24', 'D27', 'D20', 'D4'],
 'Q10': ['D10', 'D3', 'D30', 'D4', 'D7'],
 'Q11': ['D11', 'D15', 'D6', 'D20', 'D26'],
 'Q12': ['D12', 'D15', 'D6', 'D8', 'D26'],
 'Q13': ['D13', 'D11', 'D2', 'D14', 'D22'],
 'Q14': ['D14', 'D26', 'D25', 'D30', 'D5'],
 'Q15': ['D15', 'D5', 'D26', 'D14', 'D30'],
 'Q16': ['D16', 'D15', 'D5', 'D19', 'D26'],
 'Q17': ['D17', 'D15', 'D6', 'D26', 'D4'],
 'Q18': ['D18', 'D20', 'D4', 'D5', 'D15'],
 'Q19': ['D19', 'D18', 'D27', 'D23', 'D22'],
 'Q20': ['D20', 'D10', 'D27', 'D8', 'D4'],
 'Q21': ['D21', 'D5', 'D23', 'D27', 'D19'],
 'Q22': ['D22', 'D23', 'D19', 'D1', 'D27'],
 'Q23': ['D23', 'D26', 'D19', 'D12', 'D18'],
 'Q24': ['D24', 'D

## Dense retrieval
문서를 임베딩 벡터로 변환한 뒤, 벡터 간 유사도를 기준으로 문서를 찾는 방식

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

In [7]:
dense_results = {}

for idx, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']

    docs = vector_store.similarity_search(query_text, k=5)

    dense_results[qid] = [doc.metadata['doc_id'] for doc in docs]

# 각 질문에 대해 dense vector 유사도 상위 5개 문서를 저장
dense_results

{'Q1': ['D1', 'D12', 'D8', 'D2', 'D23'],
 'Q2': ['D2', 'D13', 'D27', 'D8', 'D12'],
 'Q3': ['D3', 'D17', 'D10', 'D15', 'D18'],
 'Q4': ['D4', 'D9', 'D3', 'D30', 'D16'],
 'Q5': ['D5', 'D25', 'D28', 'D14', 'D24'],
 'Q6': ['D6', 'D14', 'D25', 'D26', 'D10'],
 'Q7': ['D25', 'D7', 'D14', 'D30', 'D29'],
 'Q8': ['D8', 'D12', 'D23', 'D18', 'D15'],
 'Q9': ['D9', 'D3', 'D15', 'D19', 'D8'],
 'Q10': ['D10', 'D4', 'D14', 'D5', 'D25'],
 'Q11': ['D11', 'D19', 'D12', 'D28', 'D23'],
 'Q12': ['D12', 'D8', 'D1', 'D11', 'D13'],
 'Q13': ['D13', 'D2', 'D19', 'D14', 'D27'],
 'Q14': ['D14', 'D26', 'D6', 'D25', 'D7'],
 'Q15': ['D15', 'D14', 'D26', 'D4', 'D3'],
 'Q16': ['D16', 'D28', 'D9', 'D19', 'D8'],
 'Q17': ['D17', 'D25', 'D3', 'D10', 'D6'],
 'Q18': ['D18', 'D4', 'D27', 'D14', 'D26'],
 'Q19': ['D19', 'D18', 'D27', 'D23', 'D28'],
 'Q20': ['D20', 'D22', 'D19', 'D24', 'D23'],
 'Q21': ['D21', 'D23', 'D27', 'D18', 'D30'],
 'Q22': ['D22', 'D23', 'D19', 'D11', 'D8'],
 'Q23': ['D23', 'D24', 'D22', 'D19', 'D8'],
 'Q24'

## 연관 문서 확인

In [8]:
def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

# 질문 id별 관련 문서 목록 확인
rel_docs = {
    qid : list(parse_relevant(r).keys()) for qid, r in zip(queries_df['query_id'],queries_df['relevant_doc_ids'])
}

rel_docs

{'Q1': ['D1'],
 'Q2': ['D2'],
 'Q3': ['D3'],
 'Q4': ['D4'],
 'Q5': ['D5'],
 'Q6': ['D6'],
 'Q7': ['D7', 'D25'],
 'Q8': ['D8'],
 'Q9': ['D9'],
 'Q10': ['D10'],
 'Q11': ['D11'],
 'Q12': ['D12'],
 'Q13': ['D13', 'D11'],
 'Q14': ['D14', 'D26'],
 'Q15': ['D15'],
 'Q16': ['D16'],
 'Q17': ['D17'],
 'Q18': ['D18'],
 'Q19': ['D19'],
 'Q20': ['D20'],
 'Q21': ['D21'],
 'Q22': ['D22'],
 'Q23': ['D23'],
 'Q24': ['D24'],
 'Q25': ['D25'],
 'Q26': ['D26', 'D14'],
 'Q27': ['D27'],
 'Q28': ['D28'],
 'Q29': ['D29'],
 'Q30': ['D30', 'D4', 'D7', 'D25']}

## 검색성능 평가지표 함수

검색 최적화에서는 검색 결과를 감으로만 판단하면 안 된다. 같은 질문 세트에 대해 여러 검색 방식의 결과를 비교하려면 공통 지표가 필요하다.

**평가지표 설명**

* **Precision\@k**: 상위 k개의 검색 결과 중에 진짜 필요한 문서가 얼마나 있는지를 측정한다.
  - 예컨대 k=5일 때, 결과 5개 중 관련 문서가 2개면 Precision\@5 = 2/5 = 0.4다.

* **Recall\@k**: 전체 관련 문서 중에서 상위 k개 안에 얼마나 많이 들어왔는지를 본다.
  - 예컨대 전체 관련 문서가 4개이고, 그중 3개가 상위 5개 안에 들어오면 Recall\@5 = 3/4 = 0.75다.

* **MRR (Mean Reciprocal Rank)**:
  사용자가 제시한 여러 질의에서, 각 질의별로 “첫 번째 관련 문서”가 나온 순위의 역수를 구한 뒤 평균낸 것이다.
  **예시**

  * 질의 A: 첫 관련 문서가 2위 → RR = 1/2 = 0.5
  * 질의 B: 첫 관련 문서가 3위 → RR = 1/3 ≈ 0.333
  * 질의 C: 첫 관련 문서가 1위 → RR = 1/1 = 1
  * 이 세 질의의 MRR = (0.5 + 0.333 + 1) / 3 ≈ 0.611

* **AP (Average Precision)**:
  한 질의 결과 리스트를 순서대로 훑어가며, 관련 문서를 만날 때마다 그 시점까지의 Precision을 계산한 뒤, 관련 문서 개수로 나눈 값이다.
  **예시** (관련 문서 3개가 있고, 순위 2, 4, 5위에 위치한 경우)

  1. 2위에서 첫 관련 문서 발견 → Precision\@2 = 1/2 = 0.50
  2. 4위에서 두 번째 관련 문서 발견 → Precision\@4 = 2/4 = 0.50
  3. 5위에서 세 번째 관련 문서 발견 → Precision\@5 = 3/5 = 0.60
     AP = (0.50 + 0.50 + 0.60) / 3 ≈ 0.533

In [16]:
import numpy as np

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())

    # 상위 k개 결과만 평가 대상으로 사용한다
    top_k = predicted[:k]

    # Precision@k: 상위 k개 검색 결과 중 관련 문서가 차지하는 비율
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k

    # Recall@k: 전체 관련 문서 중 상위 k개 안에 들어온 관련 문서의 비율
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 

    # RR: 첫 번째 관련 문서가 상위 k개 안에서 몇 번째에 등장했는지 확인
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break

    # AP 관련 문서가 등장할 때마다의 Precision을 누적하고
    # 전체 관련 문서 수와 k중 작은 값으로 나눈다
    num_correct = 0
    precision_sum = 0

    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)

    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0

    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []

    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]

        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)

        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)

    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

## BM25와 Dense Retrieval 성능 비교

In [17]:
bm25_metrics = evaluate_all(bm25_results,queries_df)
dense_metrics = evaluate_all(dense_results,queries_df)

In [18]:
metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recallk@k","MRR","MAP"],
    'BM25' : [bm25_metrics["Precision@k"],bm25_metrics["Recall@k"],bm25_metrics["MRR"],bm25_metrics["MAP"]],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
})
metrics_df

,Metric,BM25,Dense
0,Precision@k,0.246667,0.233333
1,Recallk@k,1.000000,0.975000
2,MRR,0.983333,1.000000
3,MAP,0.977778,0.975000
